## Helper functions

In [1]:
results_dir = "../../results"

In [2]:
import csv
import os
import argparse
from re import X
from typing import List, Dict, Tuple, Any

from tqdm import tqdm
from collections import defaultdict
import statistics 
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.gridspec import GridSpec
import argparse
import gc
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from nnsight import LanguageModel

def load_scored_csv(csv_path: str) -> List[Dict[str, str]]:
    """
    Load CSV data with pre-computed scores.
    
    Args:
        csv_path: Path to the CSV file containing scores
    
    Returns:
        List of dictionaries containing all CSV data including scores
    """
    print(f"Loading scored data from: {csv_path}")
    
    # First pass: count rows for progress bar
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        total_rows = sum(1 for _ in reader)
    
    # Second pass: load data with progress bar
    all_rows = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in tqdm(reader, total=total_rows, desc="Loading scored data"):
            all_rows.append(row)
    
    print(f"Loaded {len(all_rows)} rows")
    return all_rows


def compute_stats_per_prompt_cot(
    scored_rows: List[Dict[str, str]]
) -> Tuple[List[List[float]], List[List[float]], List[List[str]]]:
    """
    Groups by (prompt, cot_rep_n), then nests results by prompt.
    
    Returns:
        Tuple of (means_per_prompt, std_devs_per_prompt, cots_per_prompt) where each is a 
        list of lists: [[values for prompt1's CoTs], [values for prompt2's CoTs], ...]
    """
    # Group by (prompt, cot_rep_n) to find all outputs for each CoT
    cot_groups = defaultdict(list)
    for row in tqdm(scored_rows, desc="Grouping by (prompt, cot_rep_n)"):
        key = (row['prompt'], row['cot_rep_n'])
        cot_groups[key].append(row)

    # Compute stats per (prompt, cot), organized by prompt
    # The lambda function is called: lambda: {'means': [], 'std_devs': [], 'cots': []}
    # It returns a fresh dictionary: {'means': [], 'std_devs': [], 'cots': []}
    # That dictionary is stored at prompt_to_stats['some_prompt']
    prompt_to_stats = defaultdict(lambda: {'means': [], 'std_devs': [], 'cots': []})
    
    for (prompt, cot_rep_n), rows in tqdm(cot_groups.items(), desc="Processing CoT groups"):
        chunk_scores = [float(row['strongreject_score']) for row in rows]
        
        mean = statistics.mean(chunk_scores)
        std_dev = statistics.stdev(chunk_scores) if len(chunk_scores) > 1 else 0.0
        
        # Extract the unique CoT text (should be the same for all rows in this group)
        cot_text = rows[0]['cot']
        
        prompt_to_stats[prompt]['means'].append(mean)
        prompt_to_stats[prompt]['std_devs'].append(std_dev)
        prompt_to_stats[prompt]['cots'].append(cot_text)

    # Convert to lists of lists (preserving prompt order if needed)
    means = [stats['means'] for stats in prompt_to_stats.values()]
    std_devs = [stats['std_devs'] for stats in prompt_to_stats.values()]
    cots = [stats['cots'] for stats in prompt_to_stats.values()]
    
    return means, std_devs, cots


def compute_stats_per_prompt(
    scored_rows: List[Dict[str, str]]
) -> Tuple[List[float], List[float], List[str]]:
    """
    This function groups by prompt only — aggregating across all CoTs and outputs. The standard deviation here (std_dev_i) captures total variance (from both CoT variation and output sampling).
    
    Args:
        scored_rows: List of dictionaries containing CSV data with scores
    
    Returns:
        Tuple of (means, std_devs, prompts)
    """
    means = []
    std_devs = []
    prompts = []

    # Group by prompt to find all outputs for each prompt
    prompt_groups = defaultdict(list)
    for row in tqdm(scored_rows, desc="Grouping by prompt"):
        key = row['prompt']
        prompt_groups[key].append(row)

    # Process for each prompt
    for prompt, rows in tqdm(prompt_groups.items(), desc="Processing prompt groups"):
        # Extract scores for this prompt
        chunk_scores = [float(row['strongreject_score']) for row in rows]

        mean = statistics.mean(chunk_scores)
        std_dev = statistics.stdev(chunk_scores) if len(chunk_scores) > 1 else 0.0

        # Append to lists
        means.append(mean)
        std_devs.append(std_dev)
        prompts.append(prompt)
        
    return means, std_devs, prompts

def find_quadrant(scored_rows, x_threshold=0.03, y_threshold=0.35):
    quadrant_points = []
    means_ij, std_devs_ij, cots = compute_stats_per_prompt_cot(scored_rows)
    # Average per prompt first, then overall average
    per_prompt_avgs = [statistics.mean(sublist) for sublist in std_devs_ij]
    means_i, std_devs_i, prompts = compute_stats_per_prompt(scored_rows)

    n = len(per_prompt_avgs)

    for i in range(n):
        if per_prompt_avgs[i] < x_threshold and std_devs_i[i] > y_threshold:
            dict = {
                "prompt_idx": i,
                "prompt": prompts[i],
                "cots": cots[i],
                "average mean per CoT": [f"{x:.2f}" for x in means_ij[i]],
                "std dev conditioned on CoT": [f"{x:.2f}" for x in std_devs_ij[i]],
                "std dev conditioned on prompt": f"{std_devs_i[i]:.2f}"
            }
            quadrant_points.append(dict)
    
    return quadrant_points

def set_plotting_settings():
    plt.style.use('seaborn-v0_8')
    params = {
        "ytick.color": "black",
        "xtick.color": "black",
        "axes.labelcolor": "black",
        "axes.edgecolor": "black",
        "font.family": "serif",
        "font.size": 13,
        "figure.autolayout": True,
        'figure.dpi': 600,
    }
    plt.rcParams.update(params)

    custom_colors = ['#377eb8', '#ff7f00', '#4daf4a',
                    '#f781bf', '#a65628', '#984ea3',
                    '#999999', '#e41a1c', '#dede00']
    plt.rcParams['axes.prop_cycle'] = plt.cycler(color=custom_colors)


def split_cot_into_sentences(cot: str) -> List[str]:
    """
    Split a chain-of-thought text into individual sentences.
    
    Args:
        cot: The chain-of-thought text string
        
    Returns:
        List of sentences
    """
    # Split on sentence-ending punctuation followed by whitespace
    # This regex handles . ! ? followed by whitespace or end of string
    # It also handles cases like "..." and keeps the delimiter with the sentence
    sentence_pattern = r'(?<=[.!?])\s+'
    sentences = re.split(sentence_pattern, cot.strip())
    
    # Filter out empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]
    
    return sentences


def get_sentence_token_boundaries(model, sentences: List[str]) -> List[Tuple[int, int]]:
    """
    Get the token start and end indices for each sentence.
    
    Args:
        model: The language model with tokenizer
        sentences: List of sentences
        
    Returns:
        List of (start_token_idx, end_token_idx) tuples for each sentence
    """
    boundaries = []
    current_token_idx = 0
    
    for sentence in sentences:
        sentence_tokens = model.tokenizer.encode(sentence, add_special_tokens=False)
        num_tokens = len(sentence_tokens)
        boundaries.append((current_token_idx, current_token_idx + num_tokens))
        current_token_idx += num_tokens
    
    return boundaries


def get_activations(model, quadrant_points, index_number=19, cot_number=1, layer=18):
    """Extract activations from specified layer using NNsight."""
    # Clear memory before processing
    gc.collect()
    torch.cuda.empty_cache()

    index_number -= 1
    cot_number -= 1
    prompt = quadrant_points[index_number]['prompt']
    cot = quadrant_points[index_number]['cots'][cot_number]

    # Split CoT into sentences
    sentences = split_cot_into_sentences(cot)
    
    chat = [{"role": "user", "content": prompt}]
    prompt_tokens = model.tokenizer.apply_chat_template(chat, add_generation_prompt=True)
    
    # Tokenize each sentence separately and track boundaries
    cot_tokens = []
    sentence_token_boundaries = []  # List of (start_idx, end_idx) for each sentence in cot_tokens
    
    for sentence in sentences:
        sentence_tokens = model.tokenizer.encode(sentence, add_special_tokens=False)
        start_idx = len(cot_tokens)
        cot_tokens.extend(sentence_tokens)
        end_idx = len(cot_tokens)
        sentence_token_boundaries.append((start_idx, end_idx))
    
    tokens_to_process = prompt_tokens + cot_tokens
    input_ids = torch.tensor([tokens_to_process])
    
    # Tokenize input text
    token_texts = [model.tokenizer.decode([token]) for token in tokens_to_process]
    
    # Calculate the offset for CoT tokens (after prompt tokens)
    prompt_offset = len(prompt_tokens)

    # Trace the model to get activations
    with torch.no_grad():
            with model.trace(input_ids):
                activation = model.model.layers[layer].input_layernorm.input.save()
    
    print(f"Activation tensor shape: {activation.shape}")
    print(f"Number of sentences in CoT: {len(sentences)}")
    print(f"Sentence token boundaries: {sentence_token_boundaries}")
    
    # If tensor is flattened, try to reshape it
    if len(activation.shape) == 1:
        print("len(activation_tensor.shape) == 1")
        # Calculate the expected sequence length
        seq_len = len(input_ids)
        hidden_size = model.config.hidden_size
        
        print("Tensor appears to be flattened. Attempting reshape...")
        print(f"Expected shape: [{seq_len}, {hidden_size}]")
        
        # Check if reshaping is possible
        if activation.numel() == seq_len * hidden_size:
            activation = activation.reshape(seq_len, hidden_size)
            print(f"Reshaped tensor to: {activation.shape}")
        else:
            print("WARNING: Cannot reshape tensor to expected dimensions.")
            print(f"Tensor has {activation.numel()} elements, but expected {seq_len * hidden_size}")
    
    # Clear cache after processing
    gc.collect()
    torch.cuda.empty_cache()
    
    return activation, token_texts, sentences, sentence_token_boundaries, prompt_offset

def compute_cosine_similarities(activation_tensor, direction_vector):
    """Compute cosine similarity between direction vector and each token's activation."""
    # Clear memory before computation
    gc.collect()
    torch.cuda.empty_cache()
    
    # Remove batch dimension if present
    if len(activation_tensor.shape) == 3:
        activation_tensor = activation_tensor.squeeze(0)  # Now shape is [463, 4096]
    
    # Ensure direction vector has the right shape
    direction_vector = direction_vector.reshape(1, -1)  # Shape [1, 4096]
    
    # Compute cosine similarity for each token
    similarities = []
    for token_idx in range(activation_tensor.shape[0]):
        token_activation = activation_tensor[token_idx].reshape(1, -1)  # Shape [1, 4096]
        similarity = torch.nn.functional.cosine_similarity(
            token_activation, direction_vector, dim=1
        ).item()
        similarities.append(similarity)
    
    # Clear memory after computation
    gc.collect()
    torch.cuda.empty_cache()
    
    return similarities


def compute_sentence_similarities(similarities: List[float], 
                                   sentence_token_boundaries: List[Tuple[int, int]], 
                                   prompt_offset: int) -> List[float]:
    """
    Compute average similarity for each sentence by averaging over its tokens.
    
    Args:
        similarities: List of similarities for each token
        sentence_token_boundaries: List of (start_idx, end_idx) for each sentence in CoT
        prompt_offset: Number of tokens in the prompt (before CoT starts)
        
    Returns:
        List of average similarities for each sentence
    """
    sentence_similarities = []
    
    for start_idx, end_idx in sentence_token_boundaries:
        # Adjust indices to account for prompt tokens
        global_start = prompt_offset + start_idx
        global_end = prompt_offset + end_idx
        
        # Get similarities for this sentence's tokens
        sentence_token_sims = similarities[global_start:global_end]
        
        if sentence_token_sims:
            avg_sim = np.mean(sentence_token_sims)
        else:
            avg_sim = 0.0
            
        sentence_similarities.append(avg_sim)
    
    return sentence_similarities


def plot_heatmap(token_texts, similarities, quadrant_points, output_dir, 
                 index_number=19, cot_number=1,
                 sentences=None, sentence_token_boundaries=None, prompt_offset=None):
    
    """Create a heatmap visualization of token similarities with sentence indices on x-axis."""
    plt.figure(figsize=(14, 5))
    output_path=os.path.join(output_dir, f"similarity_heatmap_index{index_number}_cot{cot_number}.png")

    idx_number = index_number - 1
    cot_num = cot_number - 1
    title = quadrant_points[idx_number]['prompt']
    score = quadrant_points[idx_number]['average mean per CoT'][cot_num]
    std_dev = quadrant_points[idx_number]['std dev conditioned on CoT'][cot_num]
    
    # Reshape similarities for heatmap (as a row)
    sim_matrix = np.array(similarities).reshape(1, -1)
    
    # Create heatmap without x-tick labels initially
    ax = sns.heatmap(sim_matrix, cmap='coolwarm_r', center=0, 
                   xticklabels=False, yticklabels=["Similarity"], 
                   vmin=-0.4, vmax=0.4)
    
    # If sentence information is provided, add sentence boundaries and labels
    if sentences is not None and sentence_token_boundaries is not None and prompt_offset is not None:
        # # Add vertical lines at sentence boundaries
        # for i, (start_idx, end_idx) in enumerate(sentence_token_boundaries):
        #     global_start = prompt_offset + start_idx
        #     # Draw vertical line at sentence start
        #     ax.axvline(x=global_start, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
        
        # Add sentence index labels at the center of each sentence
        sentence_centers = []
        sentence_labels = []
        for i, (start_idx, end_idx) in enumerate(sentence_token_boundaries):
            global_start = prompt_offset + start_idx
            global_end = prompt_offset + end_idx
            center = (global_start + global_end) / 2
            sentence_centers.append(center)
            sentence_labels.append(f"S{i+1}")
        
        # Also add special token positions for reference
        special_tokens = ['<｜User｜>', '<｜Assistant｜>', '</think>']
        special_positions = []
        special_labels_list = []
        for i, token in enumerate(token_texts):
            if token in special_tokens:
                special_positions.append(i)
                special_labels_list.append(token)
        
        # Combine sentence centers with special token positions
        all_positions = special_positions + sentence_centers
        all_labels = special_labels_list + sentence_labels
        
        ax.set_xticks(all_positions)
        ax.set_xticklabels(all_labels, rotation=90, fontsize=7)
        
        # Color special tokens differently
        for tick_label in ax.get_xticklabels():
            if tick_label.get_text() in special_tokens:
                tick_label.set_color('blue')
                tick_label.set_weight('bold')
            elif tick_label.get_text().startswith('S'):
                tick_label.set_color('green')
    else:
        # Fallback to original behavior
        custom_positions = []
        special_tokens = ['<｜User｜>', '<｜Assistant｜>', '</think>']
        
        for i, token in enumerate(token_texts):
            if token in special_tokens or ((i+1) % 5 == 0):
                custom_positions.append(i)
        
        ax.set_xticks(custom_positions)
        ax.set_xticklabels([token_texts[i] for i in custom_positions], rotation=90, fontsize=8)

        for tick_label in ax.get_xticklabels():
            if tick_label.get_text() in special_tokens:
                tick_label.set_color('blue')
                tick_label.set_weight('bold')

    plt.title(f'Prompt: {title[:80]}..., \nAverage mean per CoT: {float(score):.2f}, Std dev conditioned on CoT: {float(std_dev):.2f}, CoT rollout no.: {cot_number}', fontsize=10)
    plt.xlabel('Sentence Index')
    plt.tight_layout()
    plt.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f"Heatmap saved to {output_path}")
    plt.show()
    plt.close()
    
    # Clear memory after plotting
    gc.collect()


def plot_sentence_heatmap(sentence_similarities, sentences, quadrant_points, output_dir, 
                          index_number=19, cot_number=1):
    """
    Create a heatmap visualization with sentence indices on the x-axis.
    Each cell represents the average similarity for that sentence.
    
    Args:
        sentence_similarities: List of average similarities per sentence
        sentences: List of sentence texts
        quadrant_points: Data structure with prompt information
        output_dir: Directory to save the plot
        index_number: Index of the quadrant point (1-indexed)
        cot_number: CoT rollout number (1-indexed)
    """
    plt.figure(figsize=(max(12, len(sentences) * 0.3), 4))
    output_path = os.path.join(output_dir, f"sentence_similarity_heatmap_index{index_number}_cot{cot_number}.png")

    idx_number = index_number - 1
    cot_num = cot_number - 1
    title = quadrant_points[idx_number]['prompt']
    score = quadrant_points[idx_number]['average mean per CoT'][cot_num]
    std_dev = quadrant_points[idx_number]['std dev conditioned on CoT'][cot_num]
    
    # Reshape similarities for heatmap (as a row)
    sim_matrix = np.array(sentence_similarities).reshape(1, -1)
    
    # Create sentence index labels
    sentence_labels = [f"S{i+1}" for i in range(len(sentences))]
    
    # Create heatmap
    ax = sns.heatmap(sim_matrix, cmap='coolwarm_r', center=0, 
                     xticklabels=sentence_labels, yticklabels=["Avg Similarity"], 
                     vmin=-0.4, vmax=0.4, annot=True, fmt='.2f', annot_kws={'fontsize': 6})
    
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.title(f'Sentence-level Similarity\nPrompt: {title[:60]}...\nAvg mean per CoT: {float(score):.2f}, Std dev: {float(std_dev):.2f}, CoT rollout: {cot_number}', fontsize=10)
    plt.xlabel('Sentence Index')
    plt.tight_layout()
    plt.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f"Sentence heatmap saved to {output_path}")
    plt.show()
    plt.close()
    
    # Print sentence details
    print("\nSentence breakdown:")
    for i, (sentence, sim) in enumerate(zip(sentences, sentence_similarities)):
        print(f"  S{i+1} (sim={sim:.3f}): {sentence[:100]}{'...' if len(sentence) > 100 else ''}")
    
    gc.collect()


/workspace/adv-steer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Visualisation

In [3]:
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name="Qwen/Qwen3-8B"
model_name="openai/gpt-oss-20b"
scored_csv = "scored_train_harmful_prompts_cot5_out5[student].csv"
type = 'cot'
layer = 17
output_dir = os.path.join(results_dir, model_name, 'figures')
scored_csv_path = os.path.join(results_dir, model_name, "dataset", scored_csv)

scored_rows = load_scored_csv(scored_csv_path)
quadrant_points = find_quadrant(scored_rows)

# # Set plotting settings
# set_plotting_settings()

# # Load the pre-computed direction vector
# direction_vector = torch.load(os.path.join(results_dir, model_name, 'refusal_dir', f'refusal_dir_{type}_layer_{layer}.pt'))
# print(f"Loaded direction vector with shape: {direction_vector.shape}")

model = LanguageModel(model_name, device_map="auto")
# Memory cleanup after model loading
gc.collect()
torch.cuda.empty_cache()

Loading scored data from: ../../results/openai/gpt-oss-20b/dataset/scored_train_harmful_prompts_cot5_out5[student].csv


Loading scored data: 100%|██████████| 35490/35490 [00:00<00:00, 80111.86it/s]


Loaded 35490 rows


Processing prompt groups: 100%|██████████| 1450/1450 [00:00<00:00, 20372.70it/s]


In [4]:
index_number = 1
cot_number = 1

# Get activations with sentence information
activations, token_texts, sentences, sentence_token_boundaries, prompt_offset = get_activations(
    model, quadrant_points, index_number=index_number, cot_number=cot_number, layer=layer
)

# # Compute cosine similarities for each token
# similarities = compute_cosine_similarities(activations, direction_vector)

# # Compute average similarity per sentence
# sentence_similarities = compute_sentence_similarities(
#     similarities, sentence_token_boundaries, prompt_offset
# )

sentences

# # Plot token-level heatmap with sentence boundaries marked
# plot_heatmap(token_texts, similarities, quadrant_points, output_dir,
#              index_number=index_number, cot_number=cot_number,
#              sentences=sentences, sentence_token_boundaries=sentence_token_boundaries, 
#              prompt_offset=prompt_offset)

# # Plot sentence-level heatmap (sentence indices on x-axis)
# plot_sentence_heatmap(sentence_similarities, sentences, quadrant_points, output_dir,
#                       index_number=index_number, cot_number=cot_number)

# # Final memory cleanup
# gc.collect()
# torch.cuda.empty_cache()

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
MXFP4 quantization requires triton >= 3.4.0 and triton_kernels installed, we will default to dequantizing the model to bf16
Loading checkpoint shards:  67%|██████▋   | 2/3 [00:04<00:02,  2.07s/it]


NNsightException: 

Traceback (most recent call last):
  File "/workspace/adv-steer/.venv/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py", line 600, in from_pretrained
    return model_class.from_pretrained(
  File "/workspace/adv-steer/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py", line 316, in _wrapper
    return func(*args, **kwargs)
  File "/workspace/adv-steer/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py", line 5061, in from_pretrained
    ) = cls._load_pretrained_model(
  File "/workspace/adv-steer/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py", line 5524, in _load_pretrained_model
    _error_msgs, disk_offload_index, cpu_offload_index = load_shard_file(args)
  File "/workspace/adv-steer/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py", line 974, in load_shard_file
    disk_offload_index, cpu_offload_index = _load_state_dict_into_meta_model(
  File "/workspace/adv-steer/.venv/li

In [5]:
# index_number = 3
# cot_number = 2

# # Get activations with sentence information
# activations, token_texts, sentences, sentence_token_boundaries, prompt_offset = get_activations(
#     model, quadrant_points, index_number=index_number, cot_number=cot_number, layer=layer
# )

# # Compute cosine similarities for each token
# similarities = compute_cosine_similarities(activations, direction_vector)

# # Compute average similarity per sentence
# sentence_similarities = compute_sentence_similarities(
#     similarities, sentence_token_boundaries, prompt_offset
# )

# sentences

In [6]:
# index_number = 3
# cot_number = 3

# # Get activations with sentence information
# activations, token_texts, sentences, sentence_token_boundaries, prompt_offset = get_activations(
#     model, quadrant_points, index_number=index_number, cot_number=cot_number, layer=layer
# )

# # Compute cosine similarities for each token
# similarities = compute_cosine_similarities(activations, direction_vector)

# # Compute average similarity per sentence
# sentence_similarities = compute_sentence_similarities(
#     similarities, sentence_token_boundaries, prompt_offset
# )

# sentences

In [7]:
# index_number = 3
# cot_number = 4

# # Get activations with sentence information
# activations, token_texts, sentences, sentence_token_boundaries, prompt_offset = get_activations(
#     model, quadrant_points, index_number=index_number, cot_number=cot_number, layer=layer
# )

# # Compute cosine similarities for each token
# similarities = compute_cosine_similarities(activations, direction_vector)

# # Compute average similarity per sentence
# sentence_similarities = compute_sentence_similarities(
#     similarities, sentence_token_boundaries, prompt_offset
# )

# sentences[:2]

In [8]:
# index_number = 3
# cot_number = 1

# # Get activations with sentence information
# activations, token_texts, sentences, sentence_token_boundaries, prompt_offset = get_activations(
#     model, quadrant_points, index_number=index_number, cot_number=cot_number, layer=layer
# )

# # Compute cosine similarities for each token
# similarities = compute_cosine_similarities(activations, direction_vector)

# # Compute average similarity per sentence
# sentence_similarities = compute_sentence_similarities(
#     similarities, sentence_token_boundaries, prompt_offset
# )

# # Plot token-level heatmap with sentence boundaries marked
# plot_heatmap(token_texts, similarities, quadrant_points, output_dir,
#              index_number=index_number, cot_number=cot_number,
#              sentences=sentences, sentence_token_boundaries=sentence_token_boundaries, 
#              prompt_offset=prompt_offset)

# # Plot sentence-level heatmap (sentence indices on x-axis)
# plot_sentence_heatmap(sentence_similarities, sentences, quadrant_points, output_dir,
#                       index_number=index_number, cot_number=cot_number)

# # Final memory cleanup
# gc.collect()
# torch.cuda.empty_cache()